<table style="width: 100%;">
    <tr style="background-color: transparent;"><td>
        <img src="https://data-88e.github.io/assets/images/blue_text.png" width="250px" style="margin-left: 0;" />
    </td><td>
        <p style="text-align: right; font-size: 10pt;"><strong>Economic Models</strong>, Fall 2025<br>
            Dr. Eric Van Dusen
        </p></td></tr>
</table>

# Cobb-Douglas Regression and Penn World Table

The following code is in Pandas - a more advanced data science library you are not required to know.  
Just understanding the outputs of the following cells should be good!

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats

Read in the Penn World Table data, which contains data on GDP, capital stock, and labor for many countries over many years.  The data is in an Excel file and goes from 1950 to 2019 for 183 countries.

Reference:
>Feenstra, Robert C., Robert Inklaar and Marcel P. Timmer (2015), "The Next Generation of the Penn >World Table" American Economic Review, 105(10), 3150-3182, available for download at [www.ggdc.net/pwt](http://www.ggdc.net/pwt).


In [ ]:
pwt = pd.read_excel('pwt1001.xlsx', sheet_name='Data')
pwt

### Let's think about the structure of the data

In [ ]:
print("There are {} countries in the dataset.".format(pwt['countrycode'].nunique()))
pwt['countrycode'].unique()

In [ ]:
print("There are {} years in the dataset.".format(pwt['year'].nunique()))
pwt['year'].unique()

In [ ]:
calculated_rows = 70*193
rows = pwt.shape[0]
print("There are {} rows in the dataset.".format(rows))
print( "There are {} calculated rows in the dataset.".format(calculated_rows))

In [ ]:
#search for nans
pwt.isna().sum()

## Building a Graphs of ln(Y/L) vs ln(K/L) to estimate the Cobb-Douglas parameters



The coefficient of $x$ represents the relative impact of capital ($K$) on GDP ($Y$). Here is a brief refresher on $\alpha$ and the log form of the Cobb Douglas equation (seen above):
$$\ln{\frac{Y}{L}} = \alpha \ln{\frac{K}{L}} + A$$
$$e^{\ln{\frac{Y}{L}}} = e^{\alpha \ln{\frac{K}{L}} + A}$$
$$\frac{Y}{L} =  (\frac{K}{L})^{\alpha} * e^{A}$$
$$ Y = A K^{\alpha} L^{1 - \alpha}$$

In [ ]:
def graph_cobbs(ccodes, begin_date, end_date):
    for ccode in ccodes:
        first = pwt.loc[pwt["countrycode"] == ccode]
        second = first[ (first['year'] >= begin_date) & (first['year'] <= end_date) ]
        third = {}
        third['Y'] = second['cgdpe'] / second['cgdpe'].iloc[0]
        third['K'] = second['cn'] / second['cn'].iloc[0]
        third['L'] = second['emp'] / second['emp'].iloc[0]
        third['YL'] = third['Y'] /third['L']
        third['KL'] = third['K'] /third['L']
        third['lnYL'] = np.log(third['YL'])
        third['lnKL'] = np.log(third['KL'])
        third = pd.DataFrame(third)
        third = third.dropna()
        if(len(third)>0):
            f = plt.figure()
            ax = f.add_subplot(111)
            ax.scatter(third['lnKL'], third['lnYL'], label='')
            m, b, r_value, p_value, std_err = scipy.stats.linregress(third['lnKL'], third['lnYL'])
            ax.plot(third['lnKL'], m*third['lnKL'] + b, label='y = %.4f x + %.4f \n$R^2$ = %.4f' %(m, b, r_value**2))
            ax.legend()
            ax.set_xlabel('ln(K/L)')
            ax.set_ylabel('ln(Y/L)')
            plt.grid()
            ax.set_title(second['country'].iloc[0] + ' (' + ccode + '): '+ str(begin_date) + ' to '+ str(end_date))
            ax.text(0.0, 0.0, "Data Source: Penn World Tables", color='blue', fontstyle='italic', transform=f.transFigure)
         #   plt.savefig('Cobb-Douglas-' + ccode + '.png')
            plt.show()

In [ ]:
ccodes =['NER','SDN', 'IND', 'CHN', 'NOR', 'USA']
graph_cobbs(ccodes, 1994, 2017)

## History Matters 
Let's look at the data for Russia over two different time periods: 1994 - 2017 and 2003 - 2017  

In the first graph we graph 1990 to 2017.  In the second graph we graph 2003 to 2017.  Notice how the slope of the line changes.  This is because Russia had a major economic collapse in the early 1990s after the fall of the Soviet Union.  This shows that history matters when estimating economic models.  The more recent data is likely more relevant for understanding Russia's current economy.

In [ ]:

graph_cobbs(['RUS'], 1990, 2017)
graph_cobbs(['RUS'], 2003, 2017)

## Let's look at graphs with time on the X axis and Coobb-Douglas parameters on the Y axis



In [ ]:
#Graphs of Y and K vs time
begin_date = 1994
end_date = 2017
ccodes = pwt.countrycode.unique().tolist()
ccodes =['ZWE', 'RUS', 'CHN', 'VEN']
for ccode in ccodes:
    first = pwt.loc[pwt["countrycode"] == ccode]
    second = first[ (first['year'] >= begin_date) & (first['year'] <= end_date) ]
    third = {}
    third['year'] = second['year']
    third['Y'] = second['cgdpe'] / second['cgdpe'].iloc[0]
    third['K'] = second['cn'] / second['cn'].iloc[0]
    third['L'] = second['emp'] / second['emp'].iloc[0]
    third['YL'] = third['Y'] /third['L']
    third['KL'] = third['K'] /third['L']
    third['lnYL'] = np.log(third['YL'])
    third['lnKL'] = np.log(third['KL'])
    third = pd.DataFrame(third)
    third = third.dropna()
    if(len(third)>0):
        f = plt.figure(figsize=(10,6))
        ax = f.add_subplot(111)
        ax.scatter(third['year'], third['Y'], label='')
        m_y, b_y, r_value_y, p_value_y, std_err_y = scipy.stats.linregress(third['year'], third['Y'])
        ax.plot(third['year'], m_y*third['year'] + b_y, label='Y = %.4f x + %.4f \n$R^2$ = %.4f' %(m_y, b_y, r_value_y**2))
        ax.scatter(third['year'], third['K'], label='')
        m_k, b_k, r_value_k, p_value_k, std_err_k = scipy.stats.linregress(third['year'], third['K'])
        ax.plot(third['year'], m_k*third['year'] + b_k, label='K = %.4f x + %.4f \n$R^2$ = %.4f' %(m_k, b_k, r_value_k**2))
        ax.scatter(third['year'], third['L'], label='')
        m_l, b_l, r_value_l, p_value_l, std_err_l = scipy.stats.linregress(third['year'], third['L'])
        ax.plot(third['year'], m_l*third['year'] + b_l, label='L = %.4f x + %.4f \n$R^2$ = %.4f' %(m_l, b_l, r_value_l**2))
        ax.legend()
        ax.set_xlabel('Year')
        ax.set_ylabel('Y/K')
        plt.grid()
        ax.set_title(second['country'].iloc[0] + ' (' + ccode + '): '+ str(begin_date) + ' to '+ str(end_date))
        ax.text(0.0, 0.0, "Data Source: Penn World Tables", color='blue', fontstyle='italic', transform=f.transFigure)
       # plt.savefig('Cobb-Douglas-' + ccode + '.png')
        plt.show()